# HybridKla-source benchmark: matched internal comparison

This notebook builds a provisional, accession-grouped benchmark from the official HybridKla training workbook. PCBert-Kla, MM-Kla (token-gated), and an architecture-aligned HybridKla retraining use the same locked train, validation, and internal-test rows. The source lacks documented study provenance and an independent mouse test. Accession strings are not independently verified; negatives are putative. Results from this notebook are **internal comparisons**, not external validation or a complete answer to the reviewers' dataset concerns.


## 1. Run configuration

Preparation and integrity checks run by default. Model training is opt-in. The HybridKla timing pilot trains one ESM2 fold for one epoch and never scores the internal test. A full HybridKla run trains all eight branches in five protein-grouped folds; it is substantially more expensive than a single PCBert-Kla or MM-Kla run. Do not use internal-test outcomes to select settings.


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
REPO_BRANCH = 'main'
REPO_DIR = Path('/kaggle/working/Lysine-Lactylation-Sites-Prediction')
PROJECT_DIR = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT_ROOT = Path('/kaggle/working/hybridkla_source_results')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Set this if the official archive was attached as a private Kaggle dataset.
LOCAL_ARCHIVE = None  # e.g. Path('/kaggle/input/hybridkla-official/HybridKla.zip')
RUN_PAIRED_MODELS = False
RUN_HYBRIDKLA_TIMING = False
RUN_HYBRIDKLA_FULL = False
SEED = 42
PAIRED_EPOCHS = 30
PAIRED_PATIENCE = 6
SAVE_PAIRED_MODELS = False


## 2. Repository and GPU environment

A private clone uses the Kaggle secret `GITHUB_TOKEN` through `GIT_ASKPASS`; the token is neither printed nor embedded in the URL. The checked-out commit and hardware are saved with the results. Existing checkouts are not updated silently.


In [ ]:
import importlib.metadata
import json
import os
import subprocess
import sys
import tempfile

if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing.')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" x-access-token ;;\n  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0',
                    'MM_KLA_GITHUB_TOKEN': token})
        result = subprocess.run(['git', '-c', 'credential.helper=', 'clone', '--depth', '1',
                                 '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                                env=env, capture_output=True, text=True)
        del env, token
    if result.returncode:
        raise RuntimeError('Private clone failed. Check secret access and repository branch.')
if not (REPO_DIR / '.git').is_dir():
    raise RuntimeError('Repository path exists but is not a Git checkout.')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO_DIR, text=True).strip()
if branch != REPO_BRANCH:
    raise RuntimeError(f'Expected branch {REPO_BRANCH}, found {branch}.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
               cwd=PROJECT_DIR, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Enable Kaggle GPU acceleration.')
runtime = {'commit': commit, 'branch': branch, 'python': sys.version.split()[0],
           'gpu': [{'name': torch.cuda.get_device_name(i),
                    'memory_gb': round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2)}
                   for i in range(torch.cuda.device_count())],
           'packages': {name: importlib.metadata.version(name) for name in
                        ('torch', 'transformers', 'numpy', 'scikit-learn', 'biopython')}}
(OUTPUT_ROOT / 'runtime.json').write_text(json.dumps(runtime, indent=2) + '\n')
print(json.dumps(runtime, indent=2))


## 3. Official source and benchmark lock

The archive checksum and workbook checksum are fixed in the preparation script. Noncanonical windows are excluded because the three current model paths require canonical residues. Repeated exact sites and windows are removed before a deterministic accession-grouped 70/10/20 split. No mouse or study-held-out partition is synthesized from the available workbook.


In [ ]:
import hashlib
import urllib.request
import zipfile

ARCHIVE_SHA256 = '73ddb128c441d7b213c4636e6bb6ce34501e82143c2b9bc06ec329d68a422993'
ARCHIVE_URL = 'https://transkla.zzu.edu.cn/api/download/HybridKla.zip'
SOURCE_DIR = OUTPUT_ROOT / 'official_source'
SOURCE_DIR.mkdir(exist_ok=True)
archive = Path(LOCAL_ARCHIVE) if LOCAL_ARCHIVE else SOURCE_DIR / 'HybridKla.zip'
if not archive.is_file():
    try:
        urllib.request.urlretrieve(ARCHIVE_URL, archive)
    except Exception as exc:
        raise RuntimeError('Official download failed; attach HybridKla.zip as a private Kaggle dataset and set LOCAL_ARCHIVE.') from exc
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
if sha256(archive) != ARCHIVE_SHA256:
    raise ValueError('Official archive checksum mismatch.')
with zipfile.ZipFile(archive) as source:
    required = ('HybridKla/train/train_data.xlsx', 'HybridKla/train/top10.txt',
                'HybridKla/train/BLOSUM62R.txt')
    for name in required:
        destination = SOURCE_DIR / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            with source.open(name) as reader, destination.open('wb') as writer:
                import shutil
                shutil.copyfileobj(reader, writer)
workbook = SOURCE_DIR / required[0]
resources = SOURCE_DIR / 'HybridKla' / 'train'
lock_dir = OUTPUT_ROOT / 'locked_benchmark'
subprocess.run([sys.executable, 'scripts/lock_hybridkla_benchmark.py',
                '--workbook', str(workbook), '--output-dir', str(lock_dir),
                '--seed', str(SEED)], cwd=PROJECT_DIR, check=True)
lock = lock_dir / 'split_audit.json'
audit = json.loads(lock.read_text())
EXPECTED_MANIFEST_SHA256 = '9628cd756316dc5d6234b259d5a5f0a2364460e300ac3e6ba9d654239599a955'
if audit['model_manifest_sha256'] != EXPECTED_MANIFEST_SHA256:
    raise ValueError('HybridKla source lock differs from the audited manifest.')
print(json.dumps({'status': audit['benchmark_status'],
                  'source_audit': audit['source_audit'],
                  'partitions': audit['partitions'],
                  'manifest_sha256': audit['model_manifest_sha256']}, indent=2))


## 4. Cross-check the model input

The PCBert-Kla/MM-Kla loader independently verifies the manifest checksum, labels, canonical 51-residue windows, accession-group isolation, and per-partition counts.


In [ ]:
subprocess.run([sys.executable, 'scripts/run_replication.py',
                '--run', 'hybridkla-source-check', '--hybridkla-source-lock', str(lock)],
               cwd=PROJECT_DIR, check=True)


## 5. Matched PCBert-Kla and MM-Kla runs

Both models use ProtBert, four trainable encoder layers, SGD with learning rate 0.003, 30 requested epochs, validation AUPRC checkpoint selection, and a fixed 0.5 decision threshold. They share the exact same lock. The two available GPUs are used in parallel; a single GPU runs them sequentially.


In [ ]:
import time

def run_paired_models():
    phase = OUTPUT_ROOT / f'paired_seed_{SEED}'
    phase.mkdir(exist_ok=True)
    jobs = {}
    completed = {}
    try:
        for index, architecture in enumerate(('baseline', 'token_gated')):
            device = f'cuda:{index}' if torch.cuda.device_count() >= 2 else 'cuda:0'
            command = [sys.executable, 'scripts/run_replication.py', '--run', 'hybridkla-source',
                       '--hybridkla-source-lock', str(lock), '--architecture', architecture,
                       '--model-name', 'Rostlab/prot_bert', '--encoder-layers', '4',
                       '--epochs', str(PAIRED_EPOCHS), '--patience', str(PAIRED_PATIENCE),
                       '--batch-size', '4', '--device', device, '--optimizer', 'sgd',
                       '--learning-rate', '0.003', '--weight-decay', '0',
                       '--scheduler', 'none', '--positive-weighting', 'none',
                       '--monitor-metric', 'AUPRC', '--decision-threshold', '0.5',
                       '--seed', str(SEED), '--output-dir', str(phase)]
            if SAVE_PAIRED_MODELS:
                command.append('--save-models')
            logfile = (phase / f'{architecture}.log').open('w')
            started = time.monotonic()
            process = subprocess.Popen(command, cwd=PROJECT_DIR, stdout=logfile,
                                       stderr=subprocess.STDOUT)
            jobs[architecture] = (process, logfile, started)
            print(f'Started {architecture} on {device}', flush=True)
            if torch.cuda.device_count() == 1:
                code = process.wait()
                logfile.close()
                if code:
                    raise RuntimeError(f'{architecture} failed: ' + '\n'.join(
                        (phase / f'{architecture}.log').read_text(errors='replace').splitlines()[-30:]))
                completed[architecture] = round(time.monotonic() - started, 1)
                del jobs[architecture]
        while jobs:
            for architecture, (process, logfile, started) in list(jobs.items()):
                code = process.poll()
                if code is None:
                    continue
                logfile.close()
                if code:
                    raise RuntimeError(f'{architecture} failed: ' + '\n'.join(
                        (phase / f'{architecture}.log').read_text(errors='replace').splitlines()[-30:]))
                completed[architecture] = round(time.monotonic() - started, 1)
                del jobs[architecture]
            if jobs:
                time.sleep(10)
    finally:
        for process, logfile, _ in jobs.values():
            if process.poll() is None:
                process.terminate()
                process.wait()
            logfile.close()
    record = {'seed': SEED, 'lock_sha256': sha256(lock), 'wall_seconds': completed}
    (phase / 'run_times.json').write_text(json.dumps(record, indent=2) + '\n')
    return record

paired_report = run_paired_models() if RUN_PAIRED_MODELS else None
print(json.dumps(paired_report, indent=2) if paired_report else 'Paired training disabled.')


## 6. HybridKla architecture timing

This one-fold, one-epoch pilot fine-tunes the full ESM2 encoder on most training proteins and selects on validation. It does not train the remaining branches or score the internal test. Its measured time and peak GPU memory can inform the full-run decision; they do not establish total runtime.


In [ ]:
def run_hybridkla(mode):
    destination = OUTPUT_ROOT / f'hybridkla_{mode}_seed_{SEED}'
    destination.mkdir(exist_ok=True)
    command = [sys.executable, 'scripts/run_hybridkla_architecture.py',
               '--benchmark-lock', str(lock), '--resource-dir', str(resources),
               '--output-dir', str(destination), '--device', 'cuda:0', '--seed', str(SEED)]
    if mode == 'timing':
        command.append('--pilot')
    logfile = destination / 'run.log'
    with logfile.open('w') as handle:
        result = subprocess.run(command, cwd=PROJECT_DIR, stdout=handle,
                                stderr=subprocess.STDOUT)
    if result.returncode:
        raise RuntimeError(f'HybridKla {mode} failed: ' + '\n'.join(
            logfile.read_text(errors='replace').splitlines()[-35:]))
    return json.loads((destination / 'run_report.json').read_text())

timing_report = run_hybridkla('timing') if RUN_HYBRIDKLA_TIMING else None
print(json.dumps(timing_report, indent=2) if timing_report else 'HybridKla timing disabled.')


## 7. Full HybridKla architecture-aligned retraining

The eight source-model branches are retrained with five protein-grouped out-of-fold splits and a meta-classifier. ESM2 is fine-tuned, not frozen. Branch checkpoint selection uses the locked validation set; internal-test inference occurs only after selection. This differs from the authors' row-wise folds and implementation details and therefore must not be reported as an exact reproduction. Run only after the timing and memory check.


In [ ]:
if RUN_HYBRIDKLA_FULL and RUN_HYBRIDKLA_TIMING and timing_report:
    peak = timing_report['fold_reports']['ESM2_fold_0']['peak_cuda_gb']
    available = runtime['gpu'][0]['memory_gb']
    if peak is not None and peak > available * 0.9:
        raise RuntimeError('ESM2 pilot used over 90% of GPU memory; full run not started.')
full_report = run_hybridkla('full') if RUN_HYBRIDKLA_FULL else None
print(json.dumps(full_report, indent=2) if full_report else 'Full HybridKla training disabled.')


## 8. Internal-test summary

The table is generated only from completed runs; no metric value is hardcoded. Compare all three models only after all have completed on the same locked manifest and record the architecture and protocol differences alongside the scores. The internal test must not drive further tuning.


In [ ]:
import pandas as pd

records = []
if paired_report:
    for architecture in ('baseline', 'token_gated'):
        run_dir = OUTPUT_ROOT / f'paired_seed_{SEED}' / 'hybridkla_source' / 'runs'
        candidates = sorted(run_dir.glob('*/run_config.json')) if run_dir.exists() else []
        matches = [path.parent / 'test_metrics.json' for path in candidates
                   if json.loads(path.read_text()).get('architecture') == architecture]
        if len(matches) != 1 or not matches[0].is_file():
            raise RuntimeError(f'Expected exactly one {architecture} test metrics file under {run_dir}.')
        metrics = json.loads(matches[0].read_text())
        records.append({'architecture': architecture, 'split': 'internal_test', 'seed': SEED, **metrics})
if full_report and full_report.get('test_metrics', {}).get('internal_test'):
    records.append({'architecture': 'HybridKla architecture-aligned', 'split': 'internal_test',
                    'seed': SEED, **full_report['test_metrics']['internal_test']})
summary = pd.DataFrame.from_records(records)
if not summary.empty:
    display(summary)
    summary.to_csv(OUTPUT_ROOT / 'internal_comparison.csv', index=False)
else:
    print('No completed internal-test runs in this session.')
